[![pythonista](img/pythonista.png)](https://www.pythonista.io)

# Agregaciones.

## Configuración.

Ejecuta la siguiente celda antes que cualquier otra. Abre la base de datos del curso en modo de sólo lectura.

In [ ]:
import duckdb

%load_ext sql
%config SqlMagic.displaylimit = 0
%config SqlMagic.displaycon = False

conexion = duckdb.connect('data/thelook.duckdb', read_only=True)
%sql conexion

Las funciones de agregación resumen muchas filas en un solo valor. Son la base de cualquier indicador: total de ventas, número de clientes, ticket promedio, etc.

https://cloud.google.com/bigquery/docs/reference/standard-sql/aggregate_functions

## Funciones de agregación.

| Función | Resultado |
|---|---|
| ```COUNT(*)``` | Número de filas |
| ```COUNT(columna)``` | Número de valores **no nulos** |
| ```COUNT(DISTINCT columna)``` | Número de valores distintos |
| ```SUM()```, ```AVG()``` | Suma y promedio |
| ```MIN()```, ```MAX()``` | Mínimo y máximo |

* Resumen del catálogo de productos.

In [ ]:
%%sql
SELECT COUNT(*) AS productos,
  COUNT(DISTINCT brand) AS marcas,
  ROUND(AVG(retail_price), 2) AS precio_promedio,
  MIN(retail_price) AS precio_minimo,
  MAX(retail_price) AS precio_maximo
FROM products;

* Diferencia entre ```COUNT(*)``` y ```COUNT(columna)```: la segunda ignora los ```NULL```.

In [ ]:
%%sql
SELECT COUNT(*) AS ordenes,
  COUNT(shipped_at) AS enviadas,
  COUNT(delivered_at) AS entregadas
FROM orders;

### Conteo condicional.

Para contar sólo las filas que cumplen una condición, se combina ```COUNT()``` con una expresión ```CASE``` que regresa ```NULL``` cuando la condición no se cumple (```CASE``` se estudia a detalle más adelante).

In [ ]:
%%sql
SELECT COUNT(*) AS ordenes,
  COUNT(CASE WHEN status = 'Cancelled' THEN 1 END) AS canceladas,
  COUNT(CASE WHEN status = 'Returned' THEN 1 END) AS devueltas
FROM orders;

El estándar (*SQL:2003*) define además la cláusula ```FILTER```, más legible, que *DuckDB* y *PostgreSQL* admiten:

In [ ]:
%%sql
SELECT COUNT(*) AS ordenes,
  COUNT(*) FILTER (WHERE status = 'Cancelled') AS canceladas,
  COUNT(*) FILTER (WHERE status = 'Returned') AS devueltas
FROM orders;

> **Nota de dialecto:** *BigQuery* no admite ```FILTER```; en su lugar ofrece ```COUNTIF(condición)```. La forma con ```CASE``` funciona en todos los motores.

## Agrupamiento con ```GROUP BY```.

```GROUP BY``` divide las filas en grupos y calcula las agregaciones para cada uno.

* Número de productos y precio promedio por categoría.

In [ ]:
%%sql
SELECT category,
  COUNT(*) AS productos,
  ROUND(AVG(retail_price), 2) AS precio_promedio
FROM products
GROUP BY category
ORDER BY productos DESC;

* Se puede agrupar por varias columnas.

In [ ]:
%%sql
SELECT department, category, COUNT(*) AS productos
FROM products
GROUP BY department, category
ORDER BY department, productos DESC;

### Regla de oro de ```GROUP BY```.

**Toda columna en ```SELECT``` debe estar en ```GROUP BY``` o dentro de una función de agregación.** La siguiente consulta es inválida, porque ```name``` no está agrupada ni agregada: ¿qué nombre debería mostrar para cada categoría?

```sql
SELECT category, name, COUNT(*)
FROM products
GROUP BY category;

-- Binder Error: column "name" must appear in the GROUP BY clause
-- or must be part of an aggregate function.
```

> **Nota de dialecto:** *SQLite* y algunas configuraciones de *MySQL* aceptan esta consulta y muestran un nombre cualquiera, sin avisar. Es una de las razones para practicar con un motor apegado al estándar.

## Filtrado de grupos con ```HAVING```.

* ```WHERE``` filtra **filas** antes de agrupar.
* ```HAVING``` filtra **grupos** después de agregar.

La siguiente consulta obtiene las marcas con más de 35 productos en el departamento ```Men```.

In [ ]:
%%sql
SELECT brand, COUNT(*) AS productos
FROM products
WHERE department = 'Men'
GROUP BY brand
HAVING COUNT(*) > 35
ORDER BY productos DESC;

## Orden lógico de ejecución (completo).

```
FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → FETCH FIRST / LIMIT
```

## Indicadores de ventas.

La tabla ```order_items``` tiene una fila por artículo vendido, con su precio de venta (```sale_price```) y su estado.

* Ventas, artículos, órdenes y clientes por estado del artículo.

In [ ]:
%%sql
SELECT status,
  COUNT(*) AS articulos,
  COUNT(DISTINCT order_id) AS ordenes,
  COUNT(DISTINCT user_id) AS clientes,
  ROUND(SUM(sale_price), 2) AS ventas
FROM order_items
GROUP BY status
ORDER BY ventas DESC;

* Ticket promedio (venta promedio por orden), excluyendo cancelaciones y devoluciones. Para evitar un error de división entre cero, se usa ```NULLIF(divisor, 0)```, que convierte el cero en ```NULL```; cualquier división entre ```NULL``` da ```NULL```.

In [ ]:
%%sql
SELECT ROUND(SUM(sale_price) / NULLIF(COUNT(DISTINCT order_id), 0), 2) AS ticket_promedio
FROM order_items
WHERE status NOT IN ('Cancelled', 'Returned');

## Ejercicios.

1. ¿Cuántos clientes hay por país? Ordena de mayor a menor.
2. Para cada ```traffic_source``` de ```users```, calcula el número de clientes y su edad promedio (redondeada a un decimal).
3. ¿Qué categorías tienen un precio promedio mayor a 60? Muestra la categoría, el número de productos y el precio promedio.
4. En ```orders```, cuenta para cada ```gender``` cuántas órdenes hay en total y cuántas fueron devueltas (```Returned```), usando conteo condicional.
5. Calcula la tasa de devolución (devueltas / total) por género, en porcentaje con un decimal.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2026.</p>